In [52]:
import glob
import os
import random
import numpy as np
import csv
import json

In [53]:
class MajorityClassifier:
    def fit(self, X, y):
        counts = {}
        for label in y:
            counts[label] = counts.get(label, 0) + 1
        self.most_frequent = max(counts, key=counts.get)
        return self
 
    def predict(self, X):
        return [self.most_frequent for _ in X]   

In [55]:
def train_test_split(X, y, test_size=0.2, seed=42):
    idx = list(range(len(y)))
    random.Random(seed).shuffle(idx)
    n_test = int(len(y) * test_size)
    test, train = idx[:n_test], idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

In [56]:
def get_accuracy(y_true, y_pred):
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    return correct / len(y_true)

In [57]:
DATA_ROOT = "/path/to/autoPET"
 
# Build labels, AI helped with this snippet
X, y = [], []
for path in sorted(glob.glob(os.path.join(DATA_ROOT, "labelsTr", "*.nii.gz"))):
    mask = nib.load(path).dataobj
    X.append(os.path.basename(path))
    y.append(1 if mask[...].any() else 0)
 
print(f"Scans: {len(y)}  |  tumor: {sum(y)}  |  no tumor: {len(y) - sum(y)}") # AI for print

Scans: 0  |  tumor: 0  |  no tumor: 0


In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
#clf = MajorityClassifier().fit(X_train, y_train)
#print(f"Most frequent class: {clf.most_frequent}") # AI for print
#y_pred = clf.predict(X_test)
#print(f"Baseline Accuracy: {get_accuracy(y_test, y_pred):.4f}") # AI for print

In [59]:
# Load FDG labels (AI helped with this)
labels = {}
with open("data/fdg_metadata.csv") as f:
    for row in csv.DictReader(f):
        if row["Modality"] == "PT":
            patient = row["Subject ID"].replace("PETCT_", "")
            study = row["File Location"].split("/")[-2]
            name = f"fdg_{patient}_{study}"
            labels[name] = 0 if row["diagnosis"] == "NEGATIVE" else 1

# Load the first training/validation split
with open("data/splits_final.json") as f:
    split = json.load(f)[0]

X_train = [x for x in split["train"] if x in labels]
X_test = [x for x in split["val"] if x in labels]

y_train = [labels[x] for x in X_train]
y_test = [labels[x] for x in X_test]

# Test existing classifier
model = MajorityClassifier().fit(X_train, y_train)
predictions = model.predict(X_test)

print("Training scans:", len(X_train))
print("Testing scans:", len(X_test))
print("Accuracy:", get_accuracy(y_test, predictions))

Training scans: 822
Testing scans: 192
Accuracy: 0.4375


In [187]:
#Binary classification components. no encoder or pretrained weights included

import torch
from torch import nn


class BinaryClassificationHead(nn.Module):
    #Map [B, C, D, H, W] features (or pooled [B, C]) to [B, 1] logits.

    def __init__(self, in_features: int):
        super().__init__()
        if in_features <= 0:
            raise ValueError("in_features must be positive")
        self.network = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        if not isinstance(features, torch.Tensor):
            raise TypeError("Expected a feature tensor")

        if features.ndim != 2:
            raise ValueError("Expected [B, C] features")

        if features.shape[1] != 640:
            raise ValueError("Expected 640 input features")

        return self.network(features)

In [188]:
def train_classifier(X_train, y_train, epochs=2000):
    model = BinaryClassificationHead(in_features=640)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

    # X_train must contain encoder feature tensors of shape [640]
    features = torch.tensor(X_train, dtype=torch.float32)
    labels = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    for epoch in range(epochs):
        model.train()

        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 1000 == 0:
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

    return model

In [191]:
import numpy as np
data = np.load("features/encoder_features.npz")
X = data['features']
Y = data['lesion']
X_train = X[split == "train"]
X_test = X[split == "test"]
X_val = X[split == "val"]
Y_train = Y[split == "train"]
Y_test = Y[split == "test"]
Y_val = Y[split == "val"]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)



In [194]:
model = train_classifier(X_train_scaled, Y_train)

Epoch 1000/2000, Loss: 0.0097
Epoch 2000/2000, Loss: 0.0033


In [195]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
model.eval()
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)

with torch.no_grad():
    logits = model(X_val_tensor)
    probability = torch.sigmoid(logits)
    predictions = (probability >= 0.5).int().numpy().flatten()

print("Validation Accuracy:", accuracy_score(Y_val, predictions))
print(classification_report(Y_val, predictions))
print("Confusion Matrix:")
print(confusion_matrix(Y_val, predictions))

Validation Accuracy: 0.726984126984127
              precision    recall  f1-score   support

           0       0.58      0.59      0.59       103
           1       0.80      0.79      0.80       212

    accuracy                           0.73       315
   macro avg       0.69      0.69      0.69       315
weighted avg       0.73      0.73      0.73       315

Confusion Matrix:
[[ 61  42]
 [ 44 168]]
